# LLM-Assisted Personalized Fashion Recommendation Using User Preferences and Semantic Retrieval
**Authoritative Specification:** `End_to_End PipeLine.pdf`  
**Current Phase:** Phase 1 — Environment Setup, Dataset Acquisition & Catalog Preparation

---
### Notebook Outline
1. **Setup** *(Implemented in Phase 1)*
2. **Catalog** *(Implemented in Phase 1)*
3. **Embeddings** *(Placeholder - Phase 2)*
4. **Profiles** *(Placeholder - Phase 3)*
5. **LLM** *(Placeholder - Phase 4)*
6. **Ranking** *(Placeholder - Phase 4)*
7. **Explanations** *(Placeholder - Phase 4)*
8. **Interface** *(Placeholder - Phase 5)*
9. **Evaluation** *(Placeholder - Phase 6)*
10. **Export** *(Placeholder - Phase 7)*


## 1. Setup

In this section, we establish a reproducible runtime environment supporting both Google Colab (with Google Drive mounting) and local workstations. We detect Python and PyTorch environments, verify GPU availability, initialize workspace directories, install dependencies without conflicting with existing PyTorch installations, and configure global project paths with a fixed random seed of `42`.


In [ ]:
# 1.1 Colab Detection & Directory Initialization
import os
import sys
import shutil
from pathlib import Path

# Detect execution platform
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("[INFO] Running in Google Colab environment.")
    from google.colab import drive
    drive_mount_path = Path("/content/drive")
    if not drive_mount_path.exists():
        drive.mount("/content/drive")
    # Project paths in Colab
    WORKSPACE_ROOT = Path("/content/drive/MyDrive/fashion_project")
    LOCAL_RUNTIME_ROOT = Path("/content/fashion_project")
else:
    print("[INFO] Running in Local Workstation environment.")
    WORKSPACE_ROOT = Path(".").resolve()
    LOCAL_RUNTIME_ROOT = WORKSPACE_ROOT

# Define canonical directory hierarchy
DATA_DIR = LOCAL_RUNTIME_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
IMAGES_DIR = RAW_DATA_DIR / "images"
ARTIFACTS_DIR = LOCAL_RUNTIME_ROOT / "artifacts"
BACKUPS_DIR = LOCAL_RUNTIME_ROOT / "backups"
LOGS_DIR = LOCAL_RUNTIME_ROOT / "logs"

for directory in [RAW_DATA_DIR, IMAGES_DIR, ARTIFACTS_DIR, BACKUPS_DIR, LOGS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"[SUCCESS] Workspace initialized at: {LOCAL_RUNTIME_ROOT}")
print(f"  Artifacts directory: {ARTIFACTS_DIR}")
print(f"  Raw data directory:  {RAW_DATA_DIR}")


In [ ]:
# 1.2 System & Runtime Inspection
import platform

print("="*60)
print("RUNTIME HARDWARE & SOFTWARE SPECIFICATIONS")
print("="*60)
print(f"OS Platform:       {platform.platform()}")
print(f"Python Version:    {sys.version.split()[0]}")

# PyTorch & GPU Detection
try:
    import torch
    print(f"PyTorch Version:   {torch.__version__}")
    cuda_available = torch.cuda.is_available()
    print(f"CUDA Available:    {cuda_available}")
    if cuda_available:
        gpu_name = torch.cuda.get_device_name(0)
        gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"Assigned GPU:      {gpu_name} ({gpu_mem:.2f} GB VRAM)")
    else:
        print("Assigned GPU:      None (CPU Runtime)")
except ImportError:
    print("PyTorch Version:   Not Installed (will be loaded in PyTorch runtime)")

# System RAM Detection
def detect_ram():
    try:
        import psutil
        return psutil.virtual_memory().total / (1024**3)
    except Exception:
        pass
    if sys.platform == "win32":
        import ctypes
        class MEMORYSTATUSEX(ctypes.Structure):
            _fields_ = [("dwLength", ctypes.c_ulong), ("dwMemoryLoad", ctypes.c_ulong),
                        ("ullTotalPhys", ctypes.c_ulonglong), ("ullAvailPhys", ctypes.c_ulonglong),
                        ("ullTotalPageFile", ctypes.c_ulonglong), ("ullAvailPageFile", ctypes.c_ulonglong),
                        ("ullTotalVirtual", ctypes.c_ulonglong), ("ullAvailVirtual", ctypes.c_ulonglong),
                        ("ullAvailExtendedVirtual", ctypes.c_ulonglong)]
        stat = MEMORYSTATUSEX()
        stat.dwLength = ctypes.sizeof(MEMORYSTATUSEX)
        ctypes.windll.kernel32.GlobalMemoryStatusEx(ctypes.byref(stat))
        return stat.ullTotalPhys / (1024**3)
    elif os.path.exists("/proc/meminfo"):
        with open("/proc/meminfo") as f:
            for line in f:
                if "MemTotal" in line:
                    return int(line.split()[1]) / (1024**2)
    return 0.0

ram_gb = detect_ram()
print(f"System RAM:        {ram_gb:.2f} GB")

# Disk Space Detection
disk_total, disk_used, disk_free = shutil.disk_usage(".")
print(f"Disk Capacity:     {disk_total / (1024**3):.2f} GB total, {disk_free / (1024**3):.2f} GB free")
print("="*60)


In [ ]:
# 1.3 Dependency Installation (Colab / Local compatible)
# Preserves existing PyTorch installation as mandated by the implementation guide
REQUIRED_PACKAGES = [
    "pandas>=2.0.0",
    "numpy>=1.24.0",
    "pillow>=10.0.0",
    "pydantic>=2.10.0",
    "ipywidgets>=8.0.0"
]

if IN_COLAB:
    print("[INFO] Installing / verifying required packages in Colab...")
    get_ipython().system("pip install --quiet " + " ".join(REQUIRED_PACKAGES))
else:
    print("[INFO] Verifying packages in local environment...")
    import pandas, numpy, PIL, pydantic
    print(f"  pandas:  {pandas.__version__}")
    print(f"  numpy:   {numpy.__version__}")
    print(f"  pillow:  {PIL.__version__}")
    print(f"  pydantic:{pydantic.__version__}")


In [ ]:
# 1.4 Configuration Management & Random Seed Initialization
import json
import random
import numpy as np

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

CONFIG_PATH = LOCAL_RUNTIME_ROOT / "config.json"

config = {
    "project_title": "LLM-Assisted Personalized Fashion Recommendation Using User Preferences and Semantic Retrieval",
    "current_phase": 1,
    "random_seed": RANDOM_SEED,
    "dataset": {
        "source": "https://www.kaggle.com/datasets/paramaggarwal/fashion-product-images-small",
        "styles_csv": str(RAW_DATA_DIR / "styles.csv"),
        "images_dir": str(IMAGES_DIR)
    },
    "paths": {
        "catalog_csv": str(ARTIFACTS_DIR / "catalog.csv"),
        "catalog_ids_json": str(ARTIFACTS_DIR / "catalog_ids.json"),
        "data_audit_json": str(ARTIFACTS_DIR / "data_audit.json"),
        "catalog_manifest_json": str(ARTIFACTS_DIR / "catalog_manifest.json")
    },
    "planned_models": {
        "language_model": "Qwen/Qwen2.5-3B-Instruct",
        "embedding_model": "sentence-transformers/all-MiniLM-L6-v2"
    }
}

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print(f"[SUCCESS] Global configuration saved to: {CONFIG_PATH}")


## 2. Catalog

In this section, we acquire the Kaggle dataset (`fashion-product-images-small`), inspect and validate all 10 raw attributes, perform Pillow-based image file decoding checks, apply strict filtering (keeping only Apparel and Footwear, eliminating records with missing attributes or unusable images), and draw a frozen 10,000-item stratified sample across `articleType` groups using seed `42`.


In [ ]:
# 2.1 Dataset Acquisition & Verification
import zipfile

styles_csv_path = RAW_DATA_DIR / "styles.csv"
images_dir_path = IMAGES_DIR

# Kaggle download logic with Colab Secrets support and manual upload fallback
if not styles_csv_path.exists() or len(os.listdir(images_dir_path)) == 0:
    print("[INFO] Dataset not found in raw data directory. Initiating acquisition...")
    
    archive_path = RAW_DATA_DIR / "fashion-product-images-small.zip"
    if not archive_path.exists():
        archive_candidates = [
            Path("fashion_project/data/raw/fashion-product-images-small.zip"),
            Path("data/raw/fashion-product-images-small.zip")
        ]
        for c in archive_candidates:
            if c.exists():
                archive_path = c
                break
                
    if archive_path.exists():
        print(f"[INFO] Extracting archive from {archive_path}...")
        with zipfile.ZipFile(archive_path, "r") as z:
            z.extractall(RAW_DATA_DIR)
        print("[SUCCESS] Extraction completed.")
    elif IN_COLAB:
        try:
            from google.colab import userdata
            os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
            os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
            print("[INFO] Authenticating via Kaggle API from Colab Secrets...")
            get_ipython().system("kaggle datasets download -d paramaggarwal/fashion-product-images-small -p /content/fashion_project/data/raw --unzip")
        except Exception as e:
            print("[WARNING] Kaggle credentials not found in Colab Secrets.")
            print("Please configure KAGGLE_USERNAME and KAGGLE_KEY in Colab Secrets or upload archive manually.")
    else:
        print("[NOTICE] Please ensure styles.csv and images/ are present in data/raw/.")

# Final check
assert styles_csv_path.exists(), f"styles.csv not found at {styles_csv_path}"
assert images_dir_path.exists(), f"images directory not found at {images_dir_path}"
print(f"[SUCCESS] styles.csv verified: {os.path.getsize(styles_csv_path):,} bytes")
print(f"[SUCCESS] images directory verified: {len(os.listdir(images_dir_path)):,} files found")


In [ ]:
# 2.2 Dataset Ingestion & Column Validation
import csv
import pandas as pd

EXPECTED_COLUMNS = [
    "id", "gender", "masterCategory", "subCategory", "articleType",
    "baseColour", "season", "year", "usage", "productDisplayName"
]
NA_STRINGS = {"", "NA", "NaN", "null", "none"}

malformed_records = []
valid_rows = []

with open(styles_csv_path, mode="r", encoding="utf-8", errors="replace") as f:
    reader = csv.reader(f)
    header = next(reader)
    assert header == EXPECTED_COLUMNS, f"Header mismatch! Expected {EXPECTED_COLUMNS}, got {header}"
    
    for line_idx, row in enumerate(reader, start=2):
        if len(row) != 10:
            malformed_records.append({
                "line_number": line_idx,
                "field_count": len(row),
                "id": row[0] if len(row) > 0 else f"unknown_{line_idx}",
                "reason": "malformed_csv_token_count_mismatch"
            })
        else:
            cleaned = [val.strip() if val.strip() not in NA_STRINGS else None for val in row]
            valid_rows.append(cleaned)

df_raw = pd.DataFrame(valid_rows, columns=EXPECTED_COLUMNS)
df_raw["id"] = df_raw["id"].astype(str).str.strip()

print(f"Total rows parsed:       {len(valid_rows) + len(malformed_records):,}")
print(f"Valid 10-column records: {len(df_raw):,}")
print(f"Malformed records:       {len(malformed_records):,} (recorded in data audit)")
print(f"Duplicate product IDs:   {df_raw['id'].duplicated().sum()}")


In [ ]:
# 2.3 Exploratory Data Analysis & Attribute Inspection
print("="*60)
print("RAW DATASET ATTRIBUTE INSPECTION")
print("="*60)

# Missing Values Summary
missing_series = df_raw.isna().sum()
print("Missing values per column:")
for col, cnt in missing_series.items():
    print(f"  {col:<20}: {cnt:>5} ({cnt/len(df_raw)*100:>5.2f}%)")

print("
Distribution by masterCategory:")
print(df_raw["masterCategory"].value_counts(dropna=False))

print("
Top 10 articleTypes:")
print(df_raw["articleType"].value_counts(dropna=False).head(10))

print("
Top 10 baseColours:")
print(df_raw["baseColour"].value_counts(dropna=False).head(10))

print("
Distribution by Gender:")
print(df_raw["gender"].value_counts(dropna=False))

print("
Distribution by Usage:")
print(df_raw["usage"].value_counts(dropna=False))
print("="*60)


In [ ]:
# 2.4 Multithreaded Image Validation with Pillow
from PIL import Image
from concurrent.futures import ThreadPoolExecutor

existing_image_files = set(os.listdir(images_dir_path))
all_raw_ids = df_raw["id"].tolist()

missing_image_ids = [pid for pid in all_raw_ids if f"{pid}.jpg" not in existing_image_files]
existing_image_ids = [pid for pid in all_raw_ids if f"{pid}.jpg" in existing_image_files]

def check_image_file(pid):
    p = images_dir_path / f"{pid}.jpg"
    try:
        with Image.open(p) as img:
            img.verify()
        with Image.open(p) as img:
            img.load()
        return pid, True, None
    except Exception as e:
        return pid, False, str(e)

print(f"[INFO] Validating {len(existing_image_ids):,} images with Pillow (32 threads)...")
with ThreadPoolExecutor(max_workers=32) as executor:
    image_check_results = list(executor.map(check_image_file, existing_image_ids))

valid_image_ids = set(pid for pid, ok, err in image_check_results if ok)
corrupted_images = [{"id": pid, "error": err} for pid, ok, err in image_check_results if not ok]

print(f"  Valid decodable images:   {len(valid_image_ids):,}")
print(f"  Missing image files:      {len(missing_image_ids):,} -> {missing_image_ids}")
print(f"  Corrupted image files:    {len(corrupted_images):,}")


In [ ]:
# 2.5 Filtering, Stratified Sampling & Frozen Catalog Generation
# Rule 1: Keep masterCategory in Apparel & Footwear
af_mask = df_raw["masterCategory"].isin(["Apparel", "Footwear"])
af_df = df_raw[af_mask].copy()

# Rule 2: Exclude missing images
has_image_mask = af_df["id"].isin(valid_image_ids)
af_with_images = af_df[has_image_mask].copy()

# Rule 3: Exclude records with missing attributes in any of the 10 columns
complete_metadata_mask = ~af_with_images.isna().any(axis=1)
clean_candidates = af_with_images[complete_metadata_mask].copy()

# Rule 4: Eliminate any duplicate IDs
clean_candidates = clean_candidates.drop_duplicates(subset=["id"]).sort_values(by="id").reset_index(drop=True)

total_clean_candidates = len(clean_candidates)
print(f"Total Apparel & Footwear:     {len(af_df):,}")
print(f"Eligible clean candidates:    {total_clean_candidates:,}")

# Stratified sampling across articleType capped at 10,000 items
TARGET_SIZE = min(10000, total_clean_candidates)
groups = clean_candidates.groupby("articleType")

allocations = {}
for name, group in groups:
    share = int(np.floor(len(group) * TARGET_SIZE / total_clean_candidates))
    allocations[name] = max(1, share)

diff = TARGET_SIZE - sum(allocations.values())
sorted_groups = sorted(allocations.keys(), key=lambda k: len(groups.get_group(k)), reverse=True)
for i in range(abs(diff)):
    k = sorted_groups[i % len(sorted_groups)]
    if diff > 0:
        allocations[k] += 1
    else:
        allocations[k] -= 1

sampled_subsets = []
for name, group in groups:
    s = group.sample(n=allocations[name], random_state=RANDOM_SEED)
    sampled_subsets.append(s)

catalog_df = pd.concat(sampled_subsets, ignore_index=True)
catalog_df = catalog_df.sort_values(by="id").reset_index(drop=True)
catalog_ids = catalog_df["id"].tolist()

# Export artifacts
catalog_csv_path = ARTIFACTS_DIR / "catalog.csv"
catalog_ids_path = ARTIFACTS_DIR / "catalog_ids.json"

catalog_df.to_csv(catalog_csv_path, index=False)
with open(catalog_ids_path, "w", encoding="utf-8") as f:
    json.dump(catalog_ids, f, indent=2)

print(f"[SUCCESS] Final frozen catalog exported to: {catalog_csv_path}")
print(f"  Total items:              {len(catalog_df):,}")
print(f"  Distinct article types:   {catalog_df['articleType'].nunique():,}")
print(f"  Product IDs exported:     {catalog_ids_path}")


In [ ]:
# 2.6 Visual Verification: Display Sample Catalog Products
import matplotlib.pyplot as plt

sample_subset = catalog_df.sample(n=5, random_state=42)

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, (_, row) in zip(axes, sample_subset.iterrows()):
    img_path = IMAGES_DIR / f"{row['id']}.jpg"
    try:
        im = Image.open(img_path)
        ax.imshow(im)
    except Exception:
        ax.text(0.5, 0.5, "Image Error", ha="center")
    ax.axis("off")
    title = f"{row['articleType']} ({row['baseColour']})\n{row['usage']}\nID: {row['id']}"
    ax.set_title(title, fontsize=10)

plt.tight_layout()
plt.show()


## 3. Embeddings *(Placeholder - Phase 2)*

> [!NOTE]
> **Planned Functionality:** sentence-transformers/all-MiniLM-L6-v2 text embeddings for catalog product descriptions.  
> **Development Restriction:** As mandated by the project instructions and `End_to_End PipeLine.pdf`, this section is reserved for future implementation. Do not execute or import future phase components prematurely.


In [ ]:
# Placeholder for Section 3: Embeddings (Phase 2)
# Implementation deferred to Phase 2 according to End_to_End_PipeLine.pdf.
print("[NOTICE] Section 3 (Embeddings) will be implemented in Phase 2.")


## 4. Profiles *(Placeholder - Phase 3)*

> [!NOTE]
> **Planned Functionality:** SQLite user profiles, explicit constraints, and liked items representation.  
> **Development Restriction:** As mandated by the project instructions and `End_to_End PipeLine.pdf`, this section is reserved for future implementation. Do not execute or import future phase components prematurely.


In [ ]:
# Placeholder for Section 4: Profiles (Phase 3)
# Implementation deferred to Phase 3 according to End_to_End_PipeLine.pdf.
print("[NOTICE] Section 4 (Profiles) will be implemented in Phase 3.")


## 5. LLM *(Placeholder - Phase 4)*

> [!NOTE]
> **Planned Functionality:** Qwen2.5-3B-Instruct conversational preference extraction and Pydantic validation.  
> **Development Restriction:** As mandated by the project instructions and `End_to_End PipeLine.pdf`, this section is reserved for future implementation. Do not execute or import future phase components prematurely.


In [ ]:
# Placeholder for Section 5: LLM (Phase 4)
# Implementation deferred to Phase 4 according to End_to_End_PipeLine.pdf.
print("[NOTICE] Section 5 (LLM) will be implemented in Phase 4.")


## 6. Ranking *(Placeholder - Phase 4)*

> [!NOTE]
> **Planned Functionality:** Hard constraint candidate filtering and hybrid cosine similarity scoring (0.8 request + 0.2 profile).  
> **Development Restriction:** As mandated by the project instructions and `End_to_End PipeLine.pdf`, this section is reserved for future implementation. Do not execute or import future phase components prematurely.


In [ ]:
# Placeholder for Section 6: Ranking (Phase 4)
# Implementation deferred to Phase 4 according to End_to_End_PipeLine.pdf.
print("[NOTICE] Section 6 (Ranking) will be implemented in Phase 4.")


## 7. Explanations *(Placeholder - Phase 4)*

> [!NOTE]
> **Planned Functionality:** Grounded, evidence-based LLM recommendations and fallback metadata templates.  
> **Development Restriction:** As mandated by the project instructions and `End_to_End PipeLine.pdf`, this section is reserved for future implementation. Do not execute or import future phase components prematurely.


In [ ]:
# Placeholder for Section 7: Explanations (Phase 4)
# Implementation deferred to Phase 4 according to End_to_End_PipeLine.pdf.
print("[NOTICE] Section 7 (Explanations) will be implemented in Phase 4.")


## 8. Interface *(Placeholder - Phase 5)*

> [!NOTE]
> **Planned Functionality:** ipywidgets interactive conversational prototype with image cards and feedback controls.  
> **Development Restriction:** As mandated by the project instructions and `End_to_End PipeLine.pdf`, this section is reserved for future implementation. Do not execute or import future phase components prematurely.


In [ ]:
# Placeholder for Section 8: Interface (Phase 5)
# Implementation deferred to Phase 5 according to End_to_End_PipeLine.pdf.
print("[NOTICE] Section 8 (Interface) will be implemented in Phase 5.")


## 9. Evaluation *(Placeholder - Phase 6)*

> [!NOTE]
> **Planned Functionality:** 40-query test benchmark comparing Keyword, Semantic, Personalized, and Full Pipeline retrieval.  
> **Development Restriction:** As mandated by the project instructions and `End_to_End PipeLine.pdf`, this section is reserved for future implementation. Do not execute or import future phase components prematurely.


In [ ]:
# Placeholder for Section 9: Evaluation (Phase 6)
# Implementation deferred to Phase 6 according to End_to_End_PipeLine.pdf.
print("[NOTICE] Section 9 (Evaluation) will be implemented in Phase 6.")


## 10. Export *(Placeholder - Phase 7)*

> [!NOTE]
> **Planned Functionality:** Drive synchronization, SQLite backup, and submission packaging.  
> **Development Restriction:** As mandated by the project instructions and `End_to_End PipeLine.pdf`, this section is reserved for future implementation. Do not execute or import future phase components prematurely.


In [ ]:
# Placeholder for Section 10: Export (Phase 7)
# Implementation deferred to Phase 7 according to End_to_End_PipeLine.pdf.
print("[NOTICE] Section 10 (Export) will be implemented in Phase 7.")
